# 04 - Yogyakarta Weather Warning Report Dashboard

This notebook displays the generated LSTM Autoencoder reports in lecturer-friendly tables and charts.

Run this after:

1. `01_yogyakarta_data_audit_preprocessing.ipynb`
2. `02_yogyakarta_lstm_autoencoder_training.ipynb`
3. `03_yogyakarta_anomaly_evaluation_alerts.ipynb`

It does not retrain the model. It only reads files from `../reports/`.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..").resolve()
REPORT_DIR = PROJECT_ROOT / "reports"

ALERTS_PATH = REPORT_DIR / "alerts.csv"
ANOMALY_SCORES_PATH = REPORT_DIR / "anomaly_scores.csv"
TOP_ANOMALIES_PATH = REPORT_DIR / "top_anomalies.csv"
STATION_COUNTS_PATH = REPORT_DIR / "station_anomaly_counts.csv"
THRESHOLD_SENSITIVITY_PATH = REPORT_DIR / "threshold_sensitivity.csv"

required_files = [
    ALERTS_PATH,
    ANOMALY_SCORES_PATH,
    TOP_ANOMALIES_PATH,
    STATION_COUNTS_PATH,
    THRESHOLD_SENSITIVITY_PATH,
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Run 03_yogyakarta_anomaly_evaluation_alerts.ipynb first. Missing: "
        + ", ".join(str(path) for path in missing_files)
    )

alerts = pd.read_csv(ALERTS_PATH, parse_dates=["date"], dtype={"station_id": "string"})
anomaly_scores = pd.read_csv(ANOMALY_SCORES_PATH, parse_dates=["date"], dtype={"station_id": "string"})
top_anomalies = pd.read_csv(TOP_ANOMALIES_PATH, parse_dates=["date"], dtype={"station_id": "string"})
station_counts = pd.read_csv(STATION_COUNTS_PATH, dtype={"station_id": "string"})
threshold_sensitivity = pd.read_csv(THRESHOLD_SENSITIVITY_PATH)

print("Alerts:", alerts.shape)
print("Anomaly scores:", anomaly_scores.shape)
print("Top anomalies:", top_anomalies.shape)

## Executive Summary

In [ ]:
total_windows = len(alerts)
anomaly_rows = alerts[alerts["status"] != "NORMAL"].copy()
normal_rows = alerts[alerts["status"] == "NORMAL"].copy()

summary = pd.DataFrame(
    [
        {"metric": "Total evaluated windows", "value": total_windows},
        {"metric": "Normal windows", "value": len(normal_rows)},
        {"metric": "Anomaly windows", "value": len(anomaly_rows)},
        {"metric": "Anomaly percentage", "value": f"{len(anomaly_rows) / total_windows * 100:.2f}%"},
        {"metric": "Highest anomaly score", "value": round(alerts["anomaly_score"].max(), 4)},
        {"metric": "Highest daily rainfall (mm)", "value": round(alerts["rr_mm"].max(), 2)},
        {"metric": "Highest 3-day rainfall (mm)", "value": round(alerts["rain_3d_mm"].max(), 2)},
        {"metric": "Highest max wind", "value": round(alerts["ff_x"].max(), 2)},
    ]
)

display(summary)

## Warning Level Distribution

In [ ]:
status_order = ["NORMAL", "WASPADA", "SIAGA", "AWAS"]
status_counts = alerts["status"].value_counts().reindex(status_order, fill_value=0).reset_index()
status_counts.columns = ["status", "count"]

display(status_counts)

plt.figure(figsize=(8, 5))
sns.barplot(data=status_counts, x="status", y="count", order=status_order)
plt.title("Warning Level Distribution")
plt.xlabel("Status")
plt.ylabel("Window Count")
plt.tight_layout()
plt.show()

## Alert Type Distribution

In [ ]:
alert_type_counts = alerts["alert_type"].value_counts().reset_index()
alert_type_counts.columns = ["alert_type", "count"]

display(alert_type_counts)

plt.figure(figsize=(12, 5))
sns.barplot(data=alert_type_counts, y="alert_type", x="count")
plt.title("Alert Type Distribution")
plt.xlabel("Window Count")
plt.ylabel("Alert Type")
plt.tight_layout()
plt.show()

## Strongest Detected Anomalies

In [ ]:
columns_to_show = [
    "date",
    "station_id",
    "region_name",
    "status",
    "alert_type",
    "anomaly_score",
    "rr_mm",
    "rain_3d_mm",
    "rain_7d_mm",
    "ff_x",
    "ff_avg",
    "triggers",
]

display(top_anomalies[columns_to_show].head(20))

## Station Summary

In [ ]:
station_summary = (
    alerts.groupby(["station_id", "region_name", "status", "alert_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["station_id", "status", "alert_type"])
)

display(station_summary)

non_normal_station_summary = station_summary[station_summary["status"] != "NORMAL"]

plt.figure(figsize=(12, 6))
sns.barplot(data=non_normal_station_summary, x="station_id", y="count", hue="alert_type")
plt.title("Non-Normal Alert Types by Station")
plt.xlabel("Station ID")
plt.ylabel("Window Count")
plt.legend(title="Alert Type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Anomaly Score Timeline

In [ ]:
thresholds = {
    "p95": float(alerts["threshold_p95"].iloc[0]),
    "p99": float(alerts["threshold_p99"].iloc[0]),
    "p995": float(alerts["threshold_p995"].iloc[0]),
}

plt.figure(figsize=(15, 6))
for station_id, group in alerts.groupby("station_id"):
    plt.plot(group["date"], group["anomaly_score"], label=f"Station {station_id}", alpha=0.8)

plt.axhline(thresholds["p95"], color="orange", linestyle="--", label="P95 / WASPADA")
plt.axhline(thresholds["p99"], color="red", linestyle="--", label="P99 / SIAGA")
plt.axhline(thresholds["p995"], color="purple", linestyle="--", label="P99.5 / AWAS")
plt.title("Anomaly Score Timeline by Station")
plt.xlabel("Date")
plt.ylabel("Anomaly Score")
plt.legend()
plt.tight_layout()
plt.show()

## Rainfall and Wind Context for Anomalies

In [ ]:
if anomaly_rows.empty:
    print("No non-normal anomaly rows found.")
else:
    fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

    for station_id, group in alerts.groupby("station_id"):
        axes[0].plot(group["date"], group["rr_mm"], label=f"RR {station_id}", alpha=0.75)
        axes[1].plot(group["date"], group["rain_3d_mm"], label=f"Rain 3D {station_id}", alpha=0.75)
        axes[2].plot(group["date"], group["ff_x"], label=f"ff_x {station_id}", alpha=0.75)

    for ax in axes:
        for _, row in anomaly_rows.iterrows():
            ax.axvline(row["date"], color="red", alpha=0.08)
        ax.legend(loc="upper left")

    axes[0].set_title("Daily Rainfall with Anomaly Dates Marked")
    axes[0].set_ylabel("RR (mm)")
    axes[1].set_title("3-Day Rainfall with Anomaly Dates Marked")
    axes[1].set_ylabel("Rain 3D (mm)")
    axes[2].set_title("Maximum Wind with Anomaly Dates Marked")
    axes[2].set_ylabel("ff_x")
    axes[2].set_xlabel("Date")

    plt.tight_layout()
    plt.show()

## Filtered Tables

In [ ]:
display(anomaly_rows[columns_to_show].sort_values("anomaly_score", ascending=False))

## Report Consistency Checks

In [ ]:
checks = pd.DataFrame(
    [
        {
            "check": "NORMAL rows have empty triggers",
            "passed": int(((alerts["status"] != "NORMAL") | alerts["triggers"].fillna("").eq("")).all()),
        },
        {
            "check": "NORMAL rows have alert_type NORMAL",
            "passed": int(((alerts["status"] != "NORMAL") | alerts["alert_type"].eq("NORMAL")).all()),
        },
        {
            "check": "Non-normal rows have triggers",
            "passed": int(((alerts["status"] == "NORMAL") | alerts["triggers"].fillna("").ne("")).all()),
        },
        {
            "check": "Rainfall values are non-negative",
            "passed": int((alerts[["rr_mm", "rain_3d_mm", "rain_7d_mm"]] >= 0).all().all()),
        },
    ]
)

display(checks)
